In [1]:
import os
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

Маски не должны быть Ч/Б; фон черный, класс - какой нибудь другой, но не белый

In [ ]:
# Скрипт для перекарски белой маски в синий

pathes = Path('F:\\dataset\\masks').glob('**/*.png')
masks = [m for m in pathes]

for m in masks:
    mask_1 = cv2.imread(str(m), 0)
    mask_1_bgr = cv2.cvtColor(mask_1, cv2.COLOR_GRAY2BGR)
    mask_1_bgr[np.where((mask_1_bgr==[255, 255, 255]).all(axis=2))] = (255, 0, 0)
    cv2.imwrite(str(m), mask_1_bgr)

Кладем изображения и маски в images и masks, внутри них делим на train и val

Скрипт ниже сгенерирует необходимые для yolo таргеты в папке labels

In [ ]:
input_dir = '.\\dataset\\masks\\train'
output_dir = '.\\dataset\\labels\\train'

for j in os.listdir(input_dir):
    image_path = os.path.join(input_dir, j)
    # load the binary mask and get its contours
    mask = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    _, mask = cv2.threshold(mask, 1, 255, cv2.THRESH_BINARY)

    H, W = mask.shape
    contours, hierarchy = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # convert the contours to polygons
    polygons = []
    for cnt in contours:
        if cv2.contourArea(cnt) > 200:
            polygon = []
            for point in cnt:
                x, y = point[0]
                polygon.append(x / W)
                polygon.append(y / H)
            polygons.append(polygon)

    # print the polygons
    with open('{}.txt'.format(os.path.join(output_dir, j)[:-4]), 'w') as f:
        for polygon in polygons:
            for p_, p in enumerate(polygon):
                if p_ == len(polygon) - 1:
                    f.write('{}\n'.format(p))
                elif p_ == 0:
                    f.write('0 {} '.format(p))
                else:
                    f.write('{} '.format(p))

        f.close()

In [ ]:
model = YOLO('yolov8n-seg.pt')  # в случае добучения - указать путь до предобученной модели. базовая скачается автоматически
model.train(data='yolo8_config.yaml', epochs=50, imgsz=640)

# Папка где сохранились результаты будет указано в конце лога 

In [ ]:
# Пример придикта

odel_path = '.\\runs\\segment\\train\\weights\\last.pt'
image_path = 'some_img.png'

img = cv2.imread(image_path)
H, W, _ = img.shape

model = YOLO(model_path)

results = model(img)

for result in results:
    for j, mask in enumerate(result.masks.data):
        mask = mask.cpu().numpy() * 255
        mask = cv2.resize(mask, (W, H))
        cv2.imwrite('./output.png', mask)